# Full Insight Pipeline Clean

Notebook пересобирает CSI fraud insight pipeline с промежуточными сохранениями и явными точками обмена с внешними pipeline.

Цепочка:

```text
CSI load -> enrichment -> save for clustering -> load group_name -> LLM fraud group filter
-> save epk_id/event_dt -> load external enriched records -> row LLM reports
-> group summaries -> final Excel
```

Важно: API-ключи в notebook не используются. LLM-клиент должен быть создан снаружи, например в переменной `GigaChat_Max`.

In [ ]:
"""Файл full_insight_pipeline_clean.ipynb.

Notebook содержит функции:
- show_df: выводит форму, колонки и первые строки DataFrame.
- ensure_columns: проверяет наличие обязательных колонок.
- read_table_file: читает pickle, Excel или CSV файл.
- write_table_file: сохраняет DataFrame в pickle, Excel или CSV файл.
- build_week_windows: рассчитывает исторические недельные окна.
- normalize_type_accept: нормализует тип подтверждения CSI.
- load_csi_answers: загружает CSI-ответы из Spark.
- transform_survey_data_clean: разворачивает CSI-вопросы в широкий формат.
- load_hits_extra: загружает расширенный контекст сработок.
- coalesce_duplicate_columns: схлопывает дублирующиеся колонки после merge.
- add_business_channel: добавляет бизнес-канал.
- load_atm_mcc: загружает MCC по event_id.
- load_mcc_dictionary: загружает справочник MCC.
- enrich_mcc_names: добавляет названия MCC.
- safe_to_spark: преобразует pandas DataFrame в Spark DataFrame.
- load_transcription_context: загружает обращения и транскрибации.
- load_epk_prev_30d_full_hits: загружает предыдущие сработки за 30 дней.
- build_fraud_group_filter_messages: формирует prompt для LLM-фильтра групп.
- invoke_llm_content: вызывает LLM и возвращает текст ответа.
- parse_json_array_response: извлекает JSON-массив из ответа LLM.
- filter_fraud_groups_with_llm: размечает group_name как fraud/non-fraud.
- prepare_users_for_external_enrichment: готовит epk_id/event_dt для внешнего enrichment.
- is_empty: проверяет пустые значения.
- to_jsonable: приводит значения pandas/numpy к JSON-совместимому виду.
- safe_list: приводит значение к списку.
- normalize_amount: нормализует сумму.
- get_row_value: безопасно достает значение из строки.
- parse_hit_time: определяет время сработки.
- build_fact_pack_for_hit: собирает fact_pack по одной строке.
- build_model_inputs: собирает JSON-входы модели.
- build_case_summary_messages: формирует prompt для краткого описания.
- build_case_report_messages: формирует prompt для полного отчета.
- process_llm_rows: обрабатывает строки через LLM.
- build_group_summary_messages: формирует prompt для суммаризации группы.
- summarize_groups_with_llm: формирует общее описание по group_name.
"""

from __future__ import annotations

import json
import re
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import pyspark.sql.types as T
from IPython.display import Markdown, display
from pyspark.sql import DataFrame as SparkDataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import TimestampType
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 300)
warnings.filterwarnings("ignore")

if not hasattr(pd.DataFrame, "iteritems"):
    pd.DataFrame.iteritems = pd.DataFrame.items


## 1. Config, paths and external objects

In [ ]:
@dataclass(frozen=True)
class ExportConfig:
    """Конфигурация выгрузки базового датасета.

    Args:
        date_from: Начальная дата целевого периода в формате YYYY-MM-DD.
        date_to: Конечная дата целевого периода в формате YYYY-MM-DD.
        digest_week: Номера недель дайджеста.
        history_weeks: Количество недель истории.
        csi_table: Таблица CSI-ответов.
        hits_extra_table: Таблица расширенной информации по сработкам.
        automarking_table: Таблица автомаркировки.
        cards_event_table: Таблица card-событий.
        uko_event_table: Таблица UKO-событий.

    Returns:
        Объект с параметрами выгрузки.
    """

    date_from: str = "2026-05-01"
    date_to: str = "2026-05-15"
    digest_week: list[int] = field(default_factory=lambda: [14, 15])
    history_weeks: int = 6
    csi_table: str = "csp_repo_features3.history_csi_clients_answers_v2_129372114_129377108"
    hits_extra_table: str = "cspfs_repo_features3.hits_extra_info_129372427_view"
    automarking_table: str = "csp_repo_features.history_automarking_big_148078_155487"
    cards_event_table: str = "csp_afpc_sss_inc.cards_event"
    uko_event_table: str = "csp_afpc_sss_inc.uko_event"


EXPORT_CONFIG = ExportConfig()

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ENRICHED_FOR_CLUSTERING_PATH = OUTPUT_DIR / "01_enriched_for_clustering.pkl"
CLUSTERED_INPUT_PATH = OUTPUT_DIR / "02_clustered_from_external_pipeline.xlsx"
FRAUD_FILTERED_PATH = OUTPUT_DIR / "03_fraud_groups_filtered.pkl"
USERS_FOR_ENRICHMENT_PATH = OUTPUT_DIR / "04_users_for_external_enrichment.xlsx"
EXTERNAL_ENRICHED_INPUT_PATH = OUTPUT_DIR / "05_external_enriched_records.pkl"
ROW_REPORTS_PATH = OUTPUT_DIR / "06_case_reports.pkl"
FINAL_REPORT_PATH = OUTPUT_DIR / "07_final_insight_report.xlsx"

LLM_CLIENT_NAME = "GigaChat_Max"
RUN_LLM_GROUP_FILTER = False
RUN_LLM_ROW_REPORTS = False
RUN_LLM_GROUP_SUMMARIES = False
LLM_TEST_LIMIT: int | None = 2

# spark должен быть создан в окружении Jupyter.
# eng_pg должен быть создан пользователем, если нужен MCC-справочник.
# from sqlalchemy import create_engine
# eng_pg = create_engine("...")

print("OUTPUT_DIR:", OUTPUT_DIR.resolve())
print("LLM calls disabled:", not any([RUN_LLM_GROUP_FILTER, RUN_LLM_ROW_REPORTS, RUN_LLM_GROUP_SUMMARIES]))


In [ ]:
HITS_EXTRA_COLUMNS = [
    "epk_id", "event_id", "event_dt", "event_time", "event_channel", "sub_channel", "event_type",
    "age", "segment", "age_category", "resolution_first", "resolution_first_dttm",
    "resolution_last", "resolution_last_dttm", "surface", "atm_merchant_name",
    "merchant_info", "source_type_accept", "hits_extra_facts", "client_balance",
    "recipient_info", "scoring_oss", "previous_events_additional_info",
    "posterious_events_additional_info", "previous_events", "posterious_events",
    "transaction_amount", "policy_action", "type_accept", "rule_name", "main_rule",
]

CHANNEL_BY_SUB_CHANNEL = {
    "UFS.MOBILEAPI": "ДБО", "ISSUER_ACQUIRER": "Эмиссия", "UFS.BRANCHAPI": "ВСП",
    "ISSUER": "Эмиссия", "UFS.WEBAPI": "ДБО", "ESA.MOBILEAPI": "ДБО",
    "ISSUER_WEBACQUIRER": "Эмиссия", "ATMAPI": "ДБО", "UFS.MBKAPI": "ДБО",
    "UFS.ATMAPI": "ДБО", "UFS.OTHER": "ДБО", "ESA.WEBAPI": "ДБО", "ESA.BRANCHAPI": "ВСП",
}

MCC_NAME_OVERRIDES = {
    "6009": "Микрофинансовые организации", "3990": "Экосистема Яндекса",
    "3991": "Экосистема Сбера", "9390": "Госуслуги", "5262": "Маркетплейсы",
    "9406": "Микрофинансовые организации",
}

TEMATICS_LIST = [
    "Профиль заблокирован", "Несогласие с блокировкой карты",
    "Несогласие с блокировкой по компрометации/проверке правомерности (С/U)",
    "БК заблокирована по подозрению на мошенничество_С",
    "БК заблокирована по подозрению на мошенничество",
    "Причина блокировки (КРОМЕ 115-ФЗ)",
    "БК заблокирована по подозрению на мошенничество/для проверки правомерности",
    "Несогласие с блокировкой", "Разблокировка профиля СБОЛ", "ФРОД-МОНИТОРИНГ в СБОЛ",
    "Разблокировка профиля СБОЛ_АО", "Операция отклонена модулем Fraud Monitoring",
    "Операция отклонена модулем Fraud Monitoring_Ж",
    "Блокировка по компрометации/проверки правомерности (С/U) – несогласие",
    "АО_Операция отклонена модулем Fraud Monitoring", "Разблокировать карту",
    "Разблокировка карты", "Разблокировка счета/карты (3)", "Блокировка карты",
    "Разблокировка счета/карты (G-g)", "БК заблокирована для проверки правомерности операции",
]


## 2. Shared helpers and date windows

In [ ]:
def show_df(df: pd.DataFrame, title: str, rows: int = 3) -> None:
    """Показывает состояние DataFrame после этапа.

    Args:
        df: DataFrame для вывода.
        title: Название этапа.
        rows: Количество строк для отображения.

    Returns:
        None. Функция печатает форму, колонки и первые строки.
    """

    print(f"\n=== {title} ===")
    print(f"shape: {df.shape}")
    print(f"columns: {list(df.columns)}")
    display(df.head(rows))


def ensure_columns(df: pd.DataFrame, required_columns: list[str], stage_name: str) -> None:
    """Проверяет наличие обязательных колонок.

    Args:
        df: DataFrame для проверки.
        required_columns: Список обязательных колонок.
        stage_name: Название этапа.

    Returns:
        None. При отсутствии колонок функция вызывает ValueError.
    """

    missing = [column for column in required_columns if column not in df.columns]
    if missing:
        raise ValueError(f"{stage_name}: не хватает колонок: {missing}")


def read_table_file(path: Path) -> pd.DataFrame:
    """Читает таблицу из pickle, Excel или CSV файла.

    Args:
        path: Путь к файлу.

    Returns:
        DataFrame с содержимым файла.
    """

    suffix = path.suffix.lower()
    if suffix in {".pkl", ".pickle"}:
        return pd.read_pickle(path)
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    if suffix == ".csv":
        return pd.read_csv(path)
    raise ValueError(f"Неподдерживаемый формат файла: {path}")


def write_table_file(df: pd.DataFrame, path: Path, index: bool = False) -> None:
    """Сохраняет DataFrame в pickle, Excel или CSV файл.

    Args:
        df: DataFrame для сохранения.
        path: Путь к файлу.
        index: Нужно ли сохранять индекс.

    Returns:
        None. Функция сохраняет файл на диск.
    """

    path.parent.mkdir(parents=True, exist_ok=True)
    if path.suffix.lower() in {".pkl", ".pickle"}:
        df.to_pickle(path)
    elif path.suffix.lower() in {".xlsx", ".xls"}:
        df.to_excel(path, index=index)
    elif path.suffix.lower() == ".csv":
        df.to_csv(path, index=index)
    else:
        raise ValueError(f"Неподдерживаемый формат файла: {path}")
    print(f"Saved: {path.resolve()}")


def build_week_windows(config: ExportConfig) -> tuple[list[list[Any]], str, str, str]:
    """Рассчитывает недельные окна и технические даты.

    Args:
        config: Конфигурация выгрузки.

    Returns:
        Кортеж `(weeks_list, history_date_from, date_from_tst, date_to_tst)`.
    """

    date_to_ts = pd.to_datetime(config.date_to)
    current_week_start = (date_to_ts - pd.Timedelta(days=6)).strftime("%Y-%m-%d")
    weeks_list: list[list[Any]] = []
    for num in range(config.history_weeks, 0, -1):
        start_week = (pd.to_datetime(current_week_start) - pd.Timedelta(days=7 * num)).strftime("%Y-%m-%d")
        end_week = (date_to_ts - pd.Timedelta(days=7 * num)).strftime("%Y-%m-%d")
        weeks_list.append([start_week, end_week, config.digest_week[-1] - num])
    weeks_list.append([current_week_start, config.date_to, config.digest_week[-1]])
    history_date_from = weeks_list[0][0]
    date_from_tst = pd.to_datetime(history_date_from).strftime("%Y%m%d")
    date_to_tst = pd.to_datetime(config.date_to).strftime("%Y%m%d")
    return weeks_list, history_date_from, date_from_tst, date_to_tst


def safe_to_spark(pdf: pd.DataFrame) -> SparkDataFrame:
    """Преобразует pandas DataFrame в Spark DataFrame с явной схемой.

    Args:
        pdf: pandas DataFrame для конвертации.

    Returns:
        Spark DataFrame с предсказуемыми типами колонок.
    """

    prepared = pdf.copy()
    object_columns = prepared.select_dtypes(include=["object"]).columns
    prepared[object_columns] = prepared[object_columns].astype(str).replace("nan", None)
    fields: list[T.StructField] = []
    for column_name, dtype in prepared.dtypes.items():
        dtype_text = str(dtype).lower()
        if column_name in {"epk_id", "user_id"}:
            fields.append(T.StructField(column_name, T.LongType(), True))
        elif "int" in dtype_text:
            fields.append(T.StructField(column_name, T.LongType(), True))
        elif "float" in dtype_text:
            fields.append(T.StructField(column_name, T.DoubleType(), True))
        elif "datetime" in dtype_text:
            fields.append(T.StructField(column_name, T.TimestampType(), True))
        else:
            fields.append(T.StructField(column_name, T.StringType(), True))
    return spark.createDataFrame(prepared, schema=T.StructType(fields))


weeks_list, history_date_from, date_from_tst, date_to_tst = build_week_windows(EXPORT_CONFIG)
print("Целевой период:", EXPORT_CONFIG.date_from, EXPORT_CONFIG.date_to)
print("Историческое окно:", history_date_from, EXPORT_CONFIG.date_to)
display(pd.DataFrame(weeks_list, columns=["start_week", "end_week", "week_num"]))


### Интерактивный режим

Функции и запуск этапов разделены по ячейкам. Запускай ячейки последовательно, но между checkpoint-ячейками можно вставлять свои блоки, подменять файлы или вручную править DataFrame перед следующим этапом.

## 3. CSI load, pivot and negative filter

### 3.1 CSI functions

In [ ]:
def normalize_type_accept(column: Any) -> Any:
    """Возвращает Spark-выражение для нормализации типа подтверждения.

    Args:
        column: Spark Column `type_accept`.

    Returns:
        Spark Column с нормализованным значением.
    """

    return (
        F.when(column == "ЕРКЦ", F.lit("ЕРКЦ"))
        .when(column == "HINT Эмиссия", F.lit("HINT Карты"))
        .when(column == "HINT ДБО", F.lit("HINT ДБО"))
        .when(column == "Сотрудник ВСП", F.lit("Сотрудник ВСП"))
        .when(column == "ГПМ", F.lit("ГПМ"))
        .when(column == "IVR", F.lit("IVR"))
        .when(column == "ChipPin", F.lit("Chip+Pin"))
        .when(column == "РО/ЗРО", F.lit("РО/ЗРО"))
        .when(column == "Black List", F.lit("Black List (БА)"))
        .when(column == "БИО", F.lit("BIO"))
        .otherwise(F.lit("Не определено/None"))
    )


def load_csi_answers(config: ExportConfig, history_start: str) -> SparkDataFrame:
    """Загружает CSI-ответы клиентов.

    Args:
        config: Конфигурация выгрузки.
        history_start: Начало исторического окна.

    Returns:
        Spark DataFrame с CSI-ответами.
    """

    return (
        spark.table(config.csi_table)
        .filter(F.col("answer_date").between(history_start, config.date_to))
        .withColumn("type_accept_corrected", normalize_type_accept(F.col("type_accept")))
        .withColumn("answer_date", F.col("answer_date").cast(TimestampType()))
    )


def transform_survey_data_clean(df: pd.DataFrame, group_columns: list[str] | None = None, question_count: int = 8) -> pd.DataFrame:
    """Разворачивает CSI-вопросы в широкий формат.

    Args:
        df: pandas DataFrame с CSI-ответами.
        group_columns: Колонки, задающие один кейс.
        question_count: Максимальное количество вопросов.

    Returns:
        pandas DataFrame с одной строкой на кейс.
    """

    if group_columns is None:
        group_columns = [column for column in ["epk_id", "event_id", "answer_date"] if column in df.columns]
    if not group_columns:
        raise ValueError("Нужна хотя бы одна колонка группировки для CSI.")

    result_rows: list[dict[str, Any]] = []
    question_columns = {"question_number", "question_text", "answer_text", "comment_text"}
    for _, group in df.groupby(group_columns, dropna=False):
        base_row: dict[str, Any] = {}
        for column in df.columns:
            if column in question_columns:
                continue
            values = group[column].dropna()
            base_row[column] = values.iloc[0] if not values.empty else None
        for q_num in range(1, question_count + 1):
            q_rows = group[pd.to_numeric(group["question_number"], errors="coerce") == q_num]
            if q_rows.empty:
                base_row[f"question_{q_num}"] = None
                base_row[f"answer_{q_num}"] = None
                base_row[f"comment_{q_num}"] = None
            else:
                q_row = q_rows.iloc[0]
                base_row[f"question_{q_num}"] = q_row.get("question_text")
                base_row[f"answer_{q_num}"] = q_row.get("answer_text")
                base_row[f"comment_{q_num}"] = q_row.get("comment_text")
        result_rows.append(base_row)
    return pd.DataFrame(result_rows)


### 3.2 Run CSI load

In [ ]:
csi_raw_df = load_csi_answers(EXPORT_CONFIG, history_date_from).toPandas()
show_df(csi_raw_df, "1. CSI raw", rows=3)

clients_hits_df = transform_survey_data_clean(csi_raw_df).drop(columns=["answer_8"], errors="ignore")
clients_hits_df["event_id"] = clients_hits_df["event_id"].astype(str)
show_df(clients_hits_df, "2. CSI wide / clients_hits_df", rows=3)

clients_hits_df_neg = clients_hits_df[pd.to_numeric(clients_hits_df["answer_1"], errors="coerce") <= 3].copy()
print("Negative CSI rows:", len(clients_hits_df_neg))
display(clients_hits_df_neg.head(3))


## 4. hits_extra, channel and MCC enrichment

### 4.1 hits_extra, channel and MCC functions

In [ ]:
def load_hits_extra(config: ExportConfig, event_ids: list[str], date_from_key: str, date_to_key: str) -> pd.DataFrame:
    """Загружает расширенную информацию по сработкам.

    Args:
        config: Конфигурация выгрузки.
        event_ids: Список event_id.
        date_from_key: Начальная дата в формате YYYYMMDD.
        date_to_key: Конечная дата в формате YYYYMMDD.

    Returns:
        pandas DataFrame с полями сработки.
    """

    if not event_ids:
        return pd.DataFrame(columns=HITS_EXTRA_COLUMNS)
    available_columns = spark.read.table(config.hits_extra_table).columns
    selected_columns = [column for column in HITS_EXTRA_COLUMNS if column in available_columns]
    missing_columns = sorted(set(HITS_EXTRA_COLUMNS) - set(selected_columns))
    if missing_columns:
        print("hits_extra: отсутствующие колонки пропущены:", missing_columns)
    return (
        spark.read.table(config.hits_extra_table)
        .filter(F.col("event_dt").between(date_from_key, date_to_key))
        .filter(F.col("event_id").isin(event_ids))
        .select(*selected_columns)
        .toPandas()
    )


def coalesce_duplicate_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Схлопывает дублирующиеся колонки после merge.

    Args:
        df: DataFrame с возможными суффиксами `_x` и `_y`.

    Returns:
        DataFrame с уникальными названиями колонок.
    """

    normalized = df.copy()
    normalized.columns = normalized.columns.str.replace("_x$", "", regex=True).str.replace("_y$", "", regex=True)
    normalized = normalized.applymap(lambda value: np.nan if isinstance(value, str) and value == "" else value)
    result: dict[str, pd.Series] = {}
    for column_name in normalized.columns.unique():
        data = normalized.loc[:, normalized.columns == column_name]
        result[column_name] = data.bfill(axis=1).iloc[:, 0] if data.shape[1] > 1 else data.iloc[:, 0]
    return pd.DataFrame(result)


def add_business_channel(df: pd.DataFrame) -> pd.DataFrame:
    """Добавляет бизнес-канал по `sub_channel`.

    Args:
        df: DataFrame с колонкой `sub_channel`.

    Returns:
        DataFrame с колонкой `channel`.
    """

    result = df.copy()
    source = result["sub_channel"] if "sub_channel" in result.columns else pd.Series(index=result.index, dtype="object")
    result["channel"] = source.map(CHANNEL_BY_SUB_CHANNEL).fillna("Не определено")
    return result


def load_atm_mcc(config: ExportConfig, event_ids: list[str], date_from_key: str, date_to_key: str) -> pd.DataFrame:
    """Загружает MCC по event_id.

    Args:
        config: Конфигурация выгрузки.
        event_ids: Список event_id.
        date_from_key: Начальная дата в формате YYYYMMDD.
        date_to_key: Конечная дата в формате YYYYMMDD.

    Returns:
        DataFrame с колонками `event_id`, `mcc_code`.
    """

    if not event_ids:
        return pd.DataFrame(columns=["event_id", "mcc_code"])
    return (
        spark.table(config.automarking_table)
        .withColumn("event_dt", F.date_format(F.col("event_time"), "yyyyMMdd"))
        .filter(F.col("event_dt").between(date_from_key, date_to_key))
        .filter(F.col("event_id").isin(event_ids))
        .selectExpr("cast(event_id as string) as event_id", "cast(atm_mcc as string) as mcc_code")
        .dropDuplicates(["event_id"])
        .toPandas()
    )


def load_mcc_dictionary() -> pd.DataFrame:
    """Загружает справочник MCC.

    Args:
        Нет входных аргументов. Используется внешний объект `eng_pg`.

    Returns:
        DataFrame с колонками `mcc_code`, `mcc_name`, `description`.
    """

    if "eng_pg" not in globals():
        print("eng_pg не задан. MCC-справочник будет пустым.")
        return pd.DataFrame(columns=["mcc_code", "mcc_name", "description"])
    request = """
    SELECT a.*, b.*
    FROM selfdags.mcc_codes a
    LEFT JOIN dashboards.mcc_dictionary b
    ON CAST(a.mcc_code AS INT) = CAST(b.mcc AS INT)
    """
    mcc_pd = pd.read_sql(request, eng_pg).drop(columns=["mcc", "activity"], errors="ignore")
    mcc_pd = mcc_pd[["mcc_code", "mcc_name", "description"]].drop_duplicates()
    mcc_pd["mcc_code"] = mcc_pd["mcc_code"].astype(str)
    return mcc_pd


def enrich_mcc_names(df: pd.DataFrame, mcc_dictionary: pd.DataFrame) -> pd.DataFrame:
    """Добавляет описание MCC и бизнес-переопределения названий.

    Args:
        df: DataFrame с колонкой `mcc_code`.
        mcc_dictionary: Справочник MCC.

    Returns:
        DataFrame с `mcc_name` и `description`.
    """

    result = df.copy()
    if "mcc_code" not in result.columns:
        result["mcc_code"] = None
    result["mcc_code"] = result["mcc_code"].astype(str)
    result = result.merge(mcc_dictionary, on="mcc_code", how="left")
    for mcc_code, mcc_name in MCC_NAME_OVERRIDES.items():
        result["mcc_name"] = np.where(result["mcc_code"] == mcc_code, mcc_name, result["mcc_name"])
    return result


### 4.2 Load hits_extra

In [ ]:
event_ids_list = clients_hits_df_neg["event_id"].dropna().astype(str).unique().tolist()
hits_extra_df = load_hits_extra(EXPORT_CONFIG, event_ids_list, date_from_tst, date_to_tst)
hits_extra_df["event_id"] = hits_extra_df["event_id"].astype(str)
show_df(hits_extra_df, "3. hits_extra_df", rows=3)


### 4.3 Merge hits_extra and add channel

In [ ]:
clients_hits_df_neg = clients_hits_df_neg.merge(hits_extra_df, on="event_id", how="left")
clients_hits_df_neg = coalesce_duplicate_columns(clients_hits_df_neg)
clients_hits_df_neg = add_business_channel(clients_hits_df_neg)
show_df(clients_hits_df_neg, "4. after merge hits_extra + channel", rows=3)
display(clients_hits_df_neg["channel"].value_counts(dropna=False).rename("count").reset_index())


### 4.4 Load and merge MCC

In [ ]:
atm_mcc_df = load_atm_mcc(EXPORT_CONFIG, event_ids_list, date_from_tst, date_to_tst)
clients_hits_df_neg = clients_hits_df_neg.merge(atm_mcc_df, on="event_id", how="left")
clients_hits_df_neg = enrich_mcc_names(clients_hits_df_neg, load_mcc_dictionary())
show_df(clients_hits_df_neg, "5. after MCC enrichment", rows=3)


## 5. Optional transcriptions and previous 30d hits

### 5.1 Optional stage flags

In [ ]:
LOAD_TRANSCRIPTIONS = True
LOAD_PREV_30D_HITS = True


### 5.2 Optional enrichment functions

In [ ]:
def load_transcription_context(date_from_key: str, date_to_key: str, tematiks: list[str]) -> tuple[SparkDataFrame, SparkDataFrame]:
    """Загружает обращения и транскрибации для fraud-тематик.

    Args:
        date_from_key: Начальная дата в формате YYYYMMDD.
        date_to_key: Конечная дата в формате YYYYMMDD.
        tematiks: Список тематик обращений.

    Returns:
        Кортеж Spark DataFrame: `(case_info, transcribe)`.
    """

    case_info = (
        spark.read.table("csp_pcap1032pc_inc.dwh_stat_rb_detailed_report")
        .filter(F.col("report_dt").between(date_from_key, date_to_key))
        .filter(F.col("s_subj").isin(tematiks))
        .filter(~F.col("interaction_id").isNull())
        .selectExpr("created", "client_epk_id", "req_num", "subj", "s_subj", "kanal_reg", "interaction_id", "grp_cons_result")
    )
    transcribe = (
        spark.read.table("csp_pcap1032pc_inc.dialog_data_aggr_ext_unit_dcs")
        .filter(F.col("report_dt").between(date_from_key, date_to_key))
        .selectExpr(
            "msg_sc_call", "msg_ivr_call", "app_created", "report_dt",
            "msg_pprb_call_oth", "msg_pprb_chat_oth", "msg_sc_chat", "msg_va_chat",
            "msg_crm_call", "msg_crm_chat", "msg_re_call", "msg_pprb_chat",
            "app_row_id as interaction_id",
        )
    )
    return case_info, transcribe


def load_epk_prev_30d_full_hits(config: ExportConfig, event_ids: list[str], date_from_key: str, date_to_key: str) -> pd.DataFrame:
    """Загружает предыдущие сработки клиента за 30 дней до текущей сработки.

    Args:
        config: Конфигурация выгрузки.
        event_ids: Список целевых event_id.
        date_from_key: Начальная дата целевого периода в формате YYYYMMDD.
        date_to_key: Конечная дата целевого периода в формате YYYYMMDD.

    Returns:
        DataFrame с колонками `epk_id` и `hits_prev_30d`.
    """

    if not event_ids:
        return pd.DataFrame(columns=["epk_id", "hits_prev_30d"])
    target_meta = (
        spark.read.table(config.hits_extra_table)
        .filter(F.col("event_dt").between(date_from_key, date_to_key))
        .filter(F.col("event_id").isin(event_ids))
        .select("epk_id", "event_id", "event_dt")
        .withColumn("dt_parsed", F.to_date(F.col("event_dt").cast("string"), "yyyyMMdd"))
        .repartition("epk_id")
        .cache()
    )
    if target_meta.count() == 0:
        target_meta.unpersist()
        return pd.DataFrame(columns=["epk_id", "hits_prev_30d"])
    min_max_dt = target_meta.agg(F.min("event_dt").alias("min_dt"), F.max("event_dt").alias("max_dt")).collect()[0]
    min_dt_str, max_dt_str = str(min_max_dt["min_dt"]), str(min_max_dt["max_dt"])
    date_from_30d_int = int((datetime.strptime(min_dt_str, "%Y%m%d") - timedelta(days=30)).strftime("%Y%m%d"))
    target_epk_ids = [row[0] for row in target_meta.select("epk_id").distinct().collect()]
    available_columns = spark.read.table(config.hits_extra_table).columns
    selected_columns = [column for column in HITS_EXTRA_COLUMNS if column in available_columns]
    base_df = (
        spark.read.table(config.hits_extra_table)
        .filter(F.col("event_dt").between(date_from_30d_int, int(max_dt_str)))
        .filter(F.col("epk_id").isin(target_epk_ids))
        .select(*dict.fromkeys(["epk_id", "event_id", "event_dt", *selected_columns]))
        .withColumn("dt_parsed", F.to_date(F.col("event_dt").cast("string"), "yyyyMMdd"))
        .repartition("epk_id")
    )
    hist_struct_cols = [F.col(f"hist.{column}") for column in selected_columns if column in base_df.columns]
    history_agg = (
        target_meta.hint("broadcast").alias("curr")
        .join(base_df.alias("hist"), on="epk_id")
        .where(
            (F.col("hist.dt_parsed") >= F.date_sub(F.col("curr.dt_parsed"), 30))
            & (F.col("hist.dt_parsed") <= F.col("curr.dt_parsed"))
            & (F.col("hist.event_id") != F.col("curr.event_id"))
        )
        .groupBy("curr.epk_id")
        .agg(F.slice(F.sort_array(F.collect_list(F.struct(*hist_struct_cols)), asc=False), 1, 100).alias("hits_prev_30d"))
    )
    result_df = target_meta.select("epk_id").join(history_agg, on="epk_id", how="left").toPandas()
    target_meta.unpersist()
    return result_df


### 5.3 Run optional transcriptions

In [ ]:
if LOAD_TRANSCRIPTIONS:
    case_info_sdf, transcribe_sdf = load_transcription_context(date_from_tst, date_to_tst, TEMATICS_LIST)
    clients_hits_spark = safe_to_spark(clients_hits_df_neg)
    join_condition = clients_hits_spark.epk_id == case_info_sdf.client_epk_id
    if "event_time" in clients_hits_spark.columns:
        join_condition = join_condition & (case_info_sdf.created >= clients_hits_spark.event_time)
    clients_hits_df_neg = (
        clients_hits_spark
        .join(case_info_sdf, on=join_condition, how="left")
        .join(transcribe_sdf, on="interaction_id", how="left")
        .dropDuplicates()
        .toPandas()
    )
    show_df(clients_hits_df_neg, "6. after optional transcriptions", rows=3)
else:
    print("LOAD_TRANSCRIPTIONS=False: этап транскрибаций пропущен.")


### 5.4 Run optional previous 30d hits

In [ ]:
if LOAD_PREV_30D_HITS:
    hits_history_df = load_epk_prev_30d_full_hits(EXPORT_CONFIG, event_ids_list, date_from_tst, date_to_tst)
    clients_hits_df_neg["epk_id"] = clients_hits_df_neg["epk_id"].astype(str)
    hits_history_df["epk_id"] = hits_history_df["epk_id"].astype(str)
    clients_hits_df_neg = clients_hits_df_neg.merge(hits_history_df, on="epk_id", how="left")
    clients_hits_df_neg["hits_prev_30d"] = clients_hits_df_neg["hits_prev_30d"].apply(
        lambda value: [row.asDict() for row in value] if isinstance(value, list) else []
    )
    show_df(clients_hits_df_neg, "7. after previous 30d hits", rows=3)
else:
    print("LOAD_PREV_30D_HITS=False: этап истории сработок пропущен.")


## 6. SAVE for external clustering

После этой ячейки передай `outputs/01_enriched_for_clustering.pkl` во внешний pipeline кластеризации. Он должен вернуть исходные строки и колонку `group_name`.

In [ ]:
ensure_columns(clients_hits_df_neg, ["epk_id", "event_id", "event_dt"], "enriched_for_clustering")
write_table_file(clients_hits_df_neg, ENRICHED_FOR_CLUSTERING_PATH)
show_df(clients_hits_df_neg, "8. saved for external clustering", rows=3)
print("Передай этот файл во внешний clustering pipeline:", ENRICHED_FOR_CLUSTERING_PATH.resolve())
print("Ожидаемый результат обратно:", CLUSTERED_INPUT_PATH.resolve(), "с колонкой group_name")


## 7. EXTERNAL LOAD POINT: clustered file with group_name

In [ ]:
clustered_df = read_table_file(CLUSTERED_INPUT_PATH)
ensure_columns(clustered_df, ["group_name", "epk_id", "event_dt"], "clustered_df")
clustered_df["group_name"] = clustered_df["group_name"].astype(str).str.strip()
clustered_df = clustered_df[clustered_df["group_name"].ne("")].copy()
show_df(clustered_df, "9. clustered_df from external pipeline", rows=3)
display(clustered_df["group_name"].value_counts(dropna=False).rename("rows").reset_index().head(30))


## 8. LLM fraud group filter by group_name only

### 8.1 Fraud group filter prompts and functions

In [ ]:
FRAUD_GROUP_FILTER_SYSTEM_PROMPT = """
Ты эксперт антифрод-аналитики. Нужно определить, связана ли группа обращений напрямую с мошенничеством.
Считай fraud-related только группы, где название явно указывает на fraud monitoring, мошенничество, подозрительные операции,
компрометацию, неправомерное списание, блокировку или проверку операции из-за риска мошенничества.
Не относить к fraud-related общие сервисные, технические, тарифные, продуктовые и нейтральные жалобы без прямой fraud-связи.
Верни только валидный JSON-массив объектов без markdown.
""".strip()

FRAUD_GROUP_FILTER_HUMAN_PROMPT = """
Разметь названия групп.

Формат ответа:
[
  {"group_name": "...", "is_fraud_related": true, "reason": "короткая причина"}
]

Названия групп:
{group_names_json}
""".strip()


def build_fraud_group_filter_messages(group_names: list[str]) -> list[dict[str, str]]:
    """Формирует сообщения для LLM-фильтра fraud-групп.

    Args:
        group_names: Список строковых названий групп.

    Returns:
        Список сообщений в формате chat messages.
    """

    return [
        {"role": "system", "content": FRAUD_GROUP_FILTER_SYSTEM_PROMPT},
        {"role": "user", "content": FRAUD_GROUP_FILTER_HUMAN_PROMPT.format(group_names_json=json.dumps(group_names, ensure_ascii=False, indent=2))},
    ]


def invoke_llm_content(llm: Any, messages: list[dict[str, str]]) -> str:
    """Вызывает LLM и возвращает текст ответа.

    Args:
        llm: Chat-модель с методом `invoke`.
        messages: Сообщения для модели.

    Returns:
        Строковый контент ответа модели.
    """

    response = llm.invoke(messages)
    return getattr(response, "content", str(response))


def parse_json_array_response(content: str) -> list[dict[str, Any]]:
    """Извлекает JSON-массив объектов из ответа LLM.

    Args:
        content: Текст ответа модели.

    Returns:
        Список словарей из JSON-массива.
    """

    try:
        parsed = json.loads(content.strip())
    except json.JSONDecodeError:
        match = re.search(r"\[.*\]", content.strip(), flags=re.DOTALL)
        if not match:
            raise
        parsed = json.loads(match.group(0))
    if not isinstance(parsed, list):
        raise ValueError("Ответ LLM должен быть JSON-массивом.")
    return [item for item in parsed if isinstance(item, dict)]


def filter_fraud_groups_with_llm(group_names: list[str], llm: Any, batch_size: int = 40) -> pd.DataFrame:
    """Размечает названия групп через LLM как связанные или не связанные с fraud.

    Args:
        group_names: Уникальные названия групп.
        llm: Chat-модель с методом `invoke`.
        batch_size: Размер батча названий групп.

    Returns:
        DataFrame с колонками `group_name`, `is_fraud_related`, `reason`.
    """

    decisions: list[dict[str, Any]] = []
    for start in tqdm(range(0, len(group_names), batch_size), desc="LLM fraud group filter"):
        messages = build_fraud_group_filter_messages(group_names[start:start + batch_size])
        decisions.extend(parse_json_array_response(invoke_llm_content(llm, messages)))
    result = pd.DataFrame(decisions)
    ensure_columns(result, ["group_name", "is_fraud_related", "reason"], "fraud_group_decisions_df")
    result["group_name"] = result["group_name"].astype(str).str.strip()
    result["is_fraud_related"] = result["is_fraud_related"].astype(bool)
    return result[["group_name", "is_fraud_related", "reason"]].drop_duplicates("group_name")


### 8.2 Run fraud group filter

In [ ]:
unique_group_names = sorted(clustered_df["group_name"].dropna().astype(str).str.strip().unique().tolist())
print("Unique group_name count:", len(unique_group_names))

if RUN_LLM_GROUP_FILTER:
    fraud_group_decisions_df = filter_fraud_groups_with_llm(unique_group_names, globals()[LLM_CLIENT_NAME])
else:
    print("RUN_LLM_GROUP_FILTER=False: LLM не вызван.")
    fraud_group_decisions_df = pd.DataFrame({
        "group_name": unique_group_names,
        "is_fraud_related": False,
        "reason": "LLM-фильтр не запускался. Заполни вручную или включи RUN_LLM_GROUP_FILTER.",
    })

show_df(fraud_group_decisions_df, "10. fraud_group_decisions_df", rows=10)


In [ ]:
fraud_group_names = fraud_group_decisions_df.loc[fraud_group_decisions_df["is_fraud_related"].astype(bool), "group_name"].astype(str).tolist()
fraud_filtered_df = clustered_df[clustered_df["group_name"].isin(fraud_group_names)].copy()
write_table_file(fraud_filtered_df, FRAUD_FILTERED_PATH)
show_df(fraud_filtered_df, "11. fraud_filtered_df", rows=3)
print("Fraud groups kept:", len(fraud_group_names))
print("Rows kept:", len(fraud_filtered_df))


## 9. SAVE epk_id/event_dt for external enrichment

### 9.1 External enrichment export function

In [ ]:
def prepare_users_for_external_enrichment(df: pd.DataFrame) -> pd.DataFrame:
    """Готовит список пользователей и дат сработок для внешнего enrichment.

    Args:
        df: DataFrame после LLM-фильтра fraud-групп.

    Returns:
        DataFrame с колонками `epk_id`, `event_dt`, `event_id`, `group_name`.
    """

    ensure_columns(df, ["epk_id", "event_dt", "event_id", "group_name"], "users_for_external_enrichment")
    result = df[["epk_id", "event_dt", "event_id", "group_name"]].copy()
    for column in result.columns:
        result[column] = result[column].astype(str)
    return result.drop_duplicates().sort_values(["group_name", "epk_id", "event_dt", "event_id"]).reset_index(drop=True)


### 9.2 Save users for external enrichment

In [ ]:
users_for_external_enrichment_df = prepare_users_for_external_enrichment(fraud_filtered_df)
write_table_file(users_for_external_enrichment_df, USERS_FOR_ENRICHMENT_PATH)
show_df(users_for_external_enrichment_df, "12. users_for_external_enrichment_df", rows=10)
print("Передай этот файл во внешний enrichment pipeline:", USERS_FOR_ENRICHMENT_PATH.resolve())
print("Ожидаемый результат обратно:", EXTERNAL_ENRICHED_INPUT_PATH.resolve())


## 10. EXTERNAL LOAD POINT: enriched records

In [ ]:
external_enriched_df = read_table_file(EXTERNAL_ENRICHED_INPUT_PATH)
ensure_columns(external_enriched_df, ["epk_id", "event_dt", "event_id", "group_name"], "external_enriched_df")
external_enriched_df["epk_id"] = external_enriched_df["epk_id"].astype(str)
external_enriched_df["event_id"] = external_enriched_df["event_id"].astype(str)
external_enriched_df["group_name"] = external_enriched_df["group_name"].astype(str)
show_df(external_enriched_df, "13. external_enriched_df", rows=3)


## 11. fact_pack build

### 11.1 fact_pack helper functions

In [ ]:
def is_empty(value: Any) -> bool:
    """Проверяет значение на пустоту.

    Args:
        value: Любое значение pandas, numpy или Python.

    Returns:
        True, если значение пустое, иначе False.
    """

    if value is None:
        return True
    if isinstance(value, str):
        return value.strip().lower() in {"", "nan", "none", "null", "nat"}
    if isinstance(value, (list, tuple, dict, set)):
        return len(value) == 0
    try:
        result = pd.isna(value)
        if isinstance(result, (np.ndarray, list)):
            return False
        return bool(result)
    except Exception:
        return False


def to_jsonable(value: Any) -> Any:
    """Приводит pandas/numpy значения к JSON-совместимому виду.

    Args:
        value: Значение любого поддерживаемого типа.

    Returns:
        Значение, которое можно сериализовать в JSON.
    """

    if is_empty(value):
        return None
    if isinstance(value, pd.Timestamp):
        return value.strftime("%Y-%m-%d %H:%M:%S")
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return float(value)
    if isinstance(value, np.ndarray):
        return [to_jsonable(item) for item in value.tolist()]
    if isinstance(value, list):
        return [to_jsonable(item) for item in value]
    if isinstance(value, tuple):
        return [to_jsonable(item) for item in value]
    if isinstance(value, dict):
        return {str(key): to_jsonable(item) for key, item in value.items()}
    return value


def safe_list(value: Any) -> list[Any]:
    """Превращает значение из ячейки в список.

    Args:
        value: Значение из ячейки DataFrame.

    Returns:
        Список значений.
    """

    if is_empty(value):
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, tuple):
        return list(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, pd.Series):
        return value.tolist()
    return [value]


def normalize_amount(value: Any, divisor: int | float = 1) -> float | None:
    """Приводит сумму к рублям.

    Args:
        value: Исходное значение суммы.
        divisor: Делитель для перевода единиц измерения.

    Returns:
        Сумма в рублях или None.
    """

    if is_empty(value):
        return None
    amount = pd.to_numeric(value, errors="coerce")
    if pd.isna(amount):
        return None
    return float(amount) / divisor


def get_row_value(row: pd.Series, column: str, default: Any = None) -> Any:
    """Безопасно достает значение из строки.

    Args:
        row: Строка DataFrame.
        column: Название колонки.
        default: Значение по умолчанию.

    Returns:
        Значение колонки или default.
    """

    if column not in row.index:
        return default
    value = row[column]
    return default if is_empty(value) else value


def parse_hit_time(row: pd.Series) -> pd.Timestamp:
    """Получает точное время сработки из строки.

    Args:
        row: Строка с данными сработки.

    Returns:
        Timestamp времени сработки.
    """

    event_time = get_row_value(row, "event_time")
    if event_time is not None:
        parsed = pd.to_datetime(event_time, errors="coerce")
        if not pd.isna(parsed):
            return parsed
    event_dt = get_row_value(row, "event_dt")
    event_dt_str = str(event_dt)
    if event_dt_str.isdigit() and len(event_dt_str) == 8:
        return pd.to_datetime(event_dt_str, format="%Y%m%d", errors="coerce")
    return pd.to_datetime(event_dt, errors="coerce")


def build_fact_pack_for_hit(row: pd.Series) -> dict[str, Any]:
    """Собирает fact_pack для одной строки сработки.

    Args:
        row: Строка с данными сработки.

    Returns:
        JSON-совместимый словарь fact_pack.
    """

    hit_time = parse_hit_time(row)
    simple_fields = [
        "epk_id", "event_id", "event_dt", "group_name", "segment", "age", "event_channel",
        "channel", "sub_channel", "event_type", "transaction_amount", "surface", "atm_merchant_name",
        "mcc_code", "mcc_name", "description", "recipient_info", "client_balance",
        "policy_action", "type_accept", "resolution_last", "previous_events_additional_info",
        "hits_prev_30d", "comment_8", "previous_events", "posterious_events",
    ]
    raw_fields = {column: to_jsonable(get_row_value(row, column)) for column in simple_fields if column in row.index}
    return {
        "case_meta": {
            "epk_id": to_jsonable(get_row_value(row, "epk_id")),
            "event_id": to_jsonable(get_row_value(row, "event_id")),
            "group_name": to_jsonable(get_row_value(row, "group_name")),
            "hit_event_time": to_jsonable(hit_time),
            "analysis_goal": "Описать хронологию, текущую сработку, признаки fraud-решения и ограничения данных.",
        },
        "raw_fields": {key: value for key, value in raw_fields.items() if not is_empty(value)},
    }


def build_model_inputs(target_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Собирает fact_pack и JSON-входы модели для строк сработок.

    Args:
        target_df: DataFrame итоговых строк после внешнего enrichment.

    Returns:
        Кортеж DataFrame: `fact_packs_df` и `model_inputs_df`.
    """

    ensure_columns(target_df, ["epk_id", "event_dt", "event_id", "group_name"], "build_model_inputs")
    fact_packs: list[dict[str, Any]] = []
    model_inputs: list[dict[str, Any]] = []
    for index, row in tqdm(target_df.iterrows(), total=len(target_df), desc="Build fact_pack"):
        try:
            fact_pack = build_fact_pack_for_hit(row)
            fact_packs.append({"epk_id": row["epk_id"], "event_id": row["event_id"], "row_index": index, "fact_pack": fact_pack})
            model_inputs.append({
                "epk_id": row["epk_id"],
                "event_id": row["event_id"],
                "group_name": row["group_name"],
                "row_index": index,
                "model_input_json": json.dumps(fact_pack, ensure_ascii=False, indent=2),
            })
        except Exception as error:
            print("Ошибка сборки fact_pack:", row.get("epk_id"), row.get("event_id"), error)
    return pd.DataFrame(fact_packs), pd.DataFrame(model_inputs)


### 11.2 Build model inputs

In [ ]:
fact_packs_df, model_inputs_df = build_model_inputs(external_enriched_df)
show_df(model_inputs_df, "14. model_inputs_df", rows=3)
if len(model_inputs_df) > 0:
    print(model_inputs_df["model_input_json"].iloc[0][:3000])


## 12. LLM row report

### 12.1 Row report prompts and functions

In [ ]:
CASE_SUMMARY_SYSTEM_PROMPT = """
Ты антифрод-аналитик. По fact_pack сделай краткую сводку сработки на русском языке.
Пиши только по фактам из входных данных. Не выдумывай причины, суммы, получателей и хронологию.
""".strip()

CASE_SUMMARY_HUMAN_PROMPT = """
Сформируй краткую сводку сработки в 3-5 предложениях.

fact_pack:
{fact_pack_json}
""".strip()

CASE_REPORT_SYSTEM_PROMPT = """
Ты антифрод-аналитик. Нужно подготовить Markdown report события по fact_pack.
Используй только предоставленные факты. Отделяй факты, наблюдения и ограничения данных.
""".strip()

CASE_REPORT_HUMAN_PROMPT = """
Подготовь Markdown report:

## Краткий вывод
## Хронология и контекст
## Текущая сработка
## Комментарий клиента
## Ограничения данных

fact_pack:
{fact_pack_json}
""".strip()


def build_case_summary_messages(fact_pack_json: str) -> list[dict[str, str]]:
    """Формирует сообщения для краткого описания сработки.

    Args:
        fact_pack_json: JSON-строка fact_pack.

    Returns:
        Список сообщений для chat-модели.
    """

    return [
        {"role": "system", "content": CASE_SUMMARY_SYSTEM_PROMPT},
        {"role": "user", "content": CASE_SUMMARY_HUMAN_PROMPT.format(fact_pack_json=fact_pack_json)},
    ]


def build_case_report_messages(fact_pack_json: str) -> list[dict[str, str]]:
    """Формирует сообщения для полного отчета по сработке.

    Args:
        fact_pack_json: JSON-строка fact_pack.

    Returns:
        Список сообщений для chat-модели.
    """

    return [
        {"role": "system", "content": CASE_REPORT_SYSTEM_PROMPT},
        {"role": "user", "content": CASE_REPORT_HUMAN_PROMPT.format(fact_pack_json=fact_pack_json)},
    ]


def process_llm_rows(model_inputs: pd.DataFrame, llm: Any, max_workers: int = 4, limit: int | None = None) -> pd.DataFrame:
    """Обрабатывает строки через LLM и возвращает summary/report.

    Args:
        model_inputs: DataFrame с колонкой `model_input_json`.
        llm: Chat-модель с методом `invoke`.
        max_workers: Количество параллельных потоков.
        limit: Максимальное количество строк для тестового запуска.

    Returns:
        DataFrame с колонками `row_index`, `case_summary`, `case_report`, `llm_error`.
    """

    ensure_columns(model_inputs, ["row_index", "model_input_json"], "process_llm_rows")
    work_df = model_inputs.head(limit).copy() if limit else model_inputs.copy()

    def process_one(row: pd.Series) -> dict[str, Any]:
        """Обрабатывает одну строку через LLM.

        Args:
            row: Строка DataFrame с fact_pack JSON.

        Returns:
            Словарь с результатами LLM или ошибкой.
        """

        try:
            fact_pack_json = row["model_input_json"]
            return {
                "row_index": row["row_index"],
                "epk_id": row.get("epk_id"),
                "event_id": row.get("event_id"),
                "group_name": row.get("group_name"),
                "case_summary": invoke_llm_content(llm, build_case_summary_messages(fact_pack_json)),
                "case_report": invoke_llm_content(llm, build_case_report_messages(fact_pack_json)),
                "llm_error": None,
            }
        except Exception as error:
            return {"row_index": row.get("row_index"), "epk_id": row.get("epk_id"), "event_id": row.get("event_id"), "group_name": row.get("group_name"), "case_summary": None, "case_report": None, "llm_error": str(error)}

    results: list[dict[str, Any]] = []
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(process_one, row) for _, row in work_df.iterrows()]
        for future in tqdm(as_completed(futures), total=len(futures), desc="LLM row reports"):
            results.append(future.result())
    return pd.DataFrame(results)


### 12.2 Run row reports

In [ ]:
if RUN_LLM_ROW_REPORTS:
    row_reports_df = process_llm_rows(model_inputs_df, globals()[LLM_CLIENT_NAME], max_workers=4, limit=LLM_TEST_LIMIT)
else:
    print("RUN_LLM_ROW_REPORTS=False: LLM не вызван.")
    row_reports_df = model_inputs_df[["row_index", "epk_id", "event_id", "group_name"]].copy()
    row_reports_df["case_summary"] = None
    row_reports_df["case_report"] = None
    row_reports_df["llm_error"] = "LLM row reports не запускались."

write_table_file(row_reports_df, ROW_REPORTS_PATH)
show_df(row_reports_df, "15. row_reports_df", rows=3)


## 13. LLM group summary

### 13.1 Group summary prompts and functions

In [ ]:
# ============================================================
# ЯЧЕЙКА: ОБОБЩЕНИЕ УЖЕ ГОТОВЫХ CASE_REPORT В GROUP_SUMMARY
# ============================================================
#
# Ожидаемый вход:
#   df — DataFrame с колонкой case_report
#
# Ожидаемый выход:
#   group_summary — итоговый Markdown-отчет по группе кейсов
#   batch_summaries_df — промежуточные сводки по батчам, если кейсов много
#
# Требование:
#   llm — твой LLM-объект с методом invoke(messages)
#
# Пример:
#   group_summary, batch_summaries_df = build_group_summary_from_case_reports(
#       df=df,
#       llm=llm,
#       case_report_col="case_report",
#       case_id_col=None,
#       batch_size=8,
#   )
# ============================================================

import math
import pandas as pd


# ============================================================
# 1. НАСТРОЙКИ
# ============================================================

CASE_REPORT_COL = "case_report"

# Если в df есть колонка с id кейса, укажи ее здесь, например "case_id" или "user_id".
# Если оставить None, id будут созданы автоматически: case_001, case_002, ...
CASE_ID_COL = None

# Количество отчетов в одном батче.
# Уменьши, если отчеты большие и модель не помещает контекст.
BATCH_SIZE = 8


# ============================================================
# 2. PROMPT ДЛЯ АНАЛИЗА ОДНОЙ ГРУППЫ ОТЧЕТОВ
# ============================================================

GROUP_SUMMARY_SYSTEM_PROMPT = """
<role>
Ты — опытный аналитик антифрод-сработок и клиентского опыта.
Ты анализируешь группу Markdown-отчетов по отдельным кейсам и выделяешь:
1. проблему каждого клиента;
2. общие паттерны поведения клиентов;
3. различия между клиентами;
4. возможную общую процессную проблему;
5. ограничения интерпретации.
</role>

<task>
На вход переданы Markdown-отчеты по нескольким антифрод-сработкам.

Сравни кейсы между собой и сформируй групповой аналитический отчет.

Нужно ответить:
1. Что произошло у каждого клиента?
2. В чем была проблема каждого клиента?
3. Какие признаки поведения повторяются в группе?
4. Чем кейсы отличаются друг от друга?
5. Какую общую процессную проблему можно проверить?
6. Какие выводы подтверждены данными?
7. Какие выводы требуют дополнительных данных?
</task>

<input_contract>
Каждый кейс передан в отдельном блоке:

<case id="case_001">
...
</case>

Внутри кейса может быть:
- краткая сводка;
- полный отчет по сработке;
- карточка сработки;
- хронология до сработки;
- остановленная операция;
- хронология после сработки;
- контекст антифрод-решения;
- комментарий пользователя;
- ограничения данных;
- итог для аналитика.

Рассматривай каждый case-блок как отдельный кейс.
Сохраняй различие между клиентами и кейсами.
</input_contract>

<source_policy>
Используй только информацию из переданных Markdown-отчетов.

Каждое содержательное утверждение подкрепляй ссылкой на кейс.

Формат ссылок:
[кейс: case_001]
[кейсы: case_001, case_003]
[кейс: case_001; источник: H001]
[кейс: case_001; источники: B001, B002, A001]

Если в исходном Markdown есть evidence_id, сохраняй их в ответе.
Если evidence_id в конкретном фрагменте нет, ссылайся на case_id.

Для групповых выводов указывай список кейсов, которые подтверждают вывод.
</source_policy>

<strict_rules>
Формулируй выводы только по переданным отчетам.

Разделяй:
1. подтвержденные факты;
2. наблюдения;
3. гипотезы для проверки;
4. ограничения данных.

Проблему клиента формулируй так:

1. Если есть комментарий пользователя:
   фиксируй только то, что прямо следует из комментария.

2. Если комментария нет:
   пиши: "Проблема по комментарию не подтверждена; зафиксирована остановка операции."

3. Если есть действия после сработки:
   описывай их как наблюдение, а не как мотив клиента.

Используй осторожные формулировки:
- "группа может указывать на...";
- "можно проверить...";
- "повторяющийся признак...";
- "по переданным отчетам подтверждено...";
- "по переданным отчетам нельзя подтвердить...".

Избегай неподтвержденных утверждений:
- что клиент хотел совершить операцию;
- что клиент был вынужден действовать иначе;
- что сработка была ошибочной;
- что операция была мошеннической;
- что причина недовольства известна при отсутствии комментария;
- что правило сработало по конкретной причине, если правило не передано.
</strict_rules>

<pattern_logic>
Ищи паттерны пяти типов:

1. Паттерны точки сработки:
   - одинаковый тип операции;
   - одинаковый канал;
   - одинаковая surface;
   - одинаковая категория merchant;
   - одинаковый policy_action;
   - одинаковая резолюция;
   - одинаковое правило, если оно передано.

2. Паттерны поведения до сработки:
   - повторяющиеся пополнения;
   - похожие платежи ранее;
   - операции в той же категории;
   - внесение наличных перед операцией;
   - входы в систему перед операцией;
   - изменение канала или устройства, если передано.

3. Паттерны поведения после сработки:
   - повтор операции;
   - похожая операция после отказа;
   - переход в другой канал;
   - альтернативная операция;
   - отсутствие активности;
   - продолжение обычных операций.

4. Паттерны пользовательских комментариев:
   - операция моя;
   - непонятна причина отказа;
   - не удалось совершить платеж;
   - повторная проблема;
   - срочная операция;
   - комментарий отсутствует.

5. Паттерны ограничений:
   - нет комментария;
   - нет получателя;
   - нет баланса;
   - нет точного правила;
   - недостаточно событий после сработки.
</pattern_logic>

<pattern_threshold>
Паттерн считается групповым, если он подтверждается минимум двумя кейсами.

Если признак встречается только в одном кейсе, называй его "единичное наблюдение".
</pattern_threshold>

<output_format>
Верни Markdown без JSON-обертки и без markdown code fence.

Структура ответа:

# Групповой анализ сработок

## 1. Краткий вывод

Напиши 4–8 предложений:
- сколько кейсов передано;
- какая общая точка сработки видна, если она есть;
- какие общие паттерны поведения видны;
- насколько уверенно можно говорить об общей проблеме;
- какие ограничения важны.

Каждое утверждение подкрепляй ссылками на кейсы.

## 2. Проблема каждого клиента

Таблица:
| Кейс | Что было остановлено | Комментарий пользователя | Наблюдаемая проблема клиента | Что было до сработки | Что было после сработки | Ограничения |
|---|---|---|---|---|---|---|

## 3. Общие паттерны поведения

Для каждого паттерна используй структуру:

### Паттерн N. Название паттерна

| Поле | Значение |
|---|---|
| Тип паттерна | точка сработки / до сработки / после сработки / комментарий / ограничение |
| Описание | ... |
| Кейсы, где встречается | ... |
| Подтверждающие факты | ... |
| Насколько устойчив | высокий / средний / низкий |
| Ограничения | ... |

После таблицы дай короткое объяснение паттерна.

## 4. Различия между кейсами

Таблица:
| Признак | Варианты в группе | Кейсы | Что это меняет для интерпретации |
|---|---|---|---|

Отрази различия:
- по операции;
- по каналу;
- по merchant/category;
- по сумме;
- по действиям до сработки;
- по действиям после сработки;
- по комментарию;
- по ограничениям данных.

## 5. Общая процессная проблема

### 5.1. Что подтверждено

Список фактов, которые повторяются в группе.
Каждый пункт со ссылками на кейсы.

### 5.2. Возможная процессная проблема для проверки

Сформулируй осторожно:
"Можно проверить, что ..."
"Группа может указывать на ..."
"Возможная зона процесса: ..."

### 5.3. Что не подтверждено

Список выводов, которые нельзя делать по этим данным.

## 6. Сегменты группы

Если внутри группы видны подгруппы, выдели их.

Для каждой подгруппы:
- название;
- какие кейсы входят;
- общий признак;
- чем отличается от других подгрупп;
- возможная интерпретация.

Если подгрупп нет, напиши:
"Явные подгруппы по переданным описаниям не выделяются."

## 7. Что проверить аналитически

Список проверок:
- какую метрику посчитать;
- по каким полям сгруппировать;
- какие данные нужны;
- какой результат подтвердит или опровергнет гипотезу.

## 8. Итоговая формулировка для отчета

Дай 1–2 аккуратных абзаца, которые можно вставить в итоговый отчет для руководителя.

Формулировка должна быть осторожной:
- без утверждения об ошибке антифрода;
- без утверждения о мотивах клиента;
- с указанием ограничений;
- с описанием проверяемой процессной зоны.
</output_format>

<style>
Пиши на русском языке.
Стиль — деловой, аналитический, спокойный.
Сохраняй формулировки короткими и проверяемыми.
</style>
""".strip()


GROUP_SUMMARY_HUMAN_PROMPT = """
<task>
Проанализируй группу Markdown-отчетов по антифрод-сработкам.

Нужно выделить:
1. в чем была проблема у каждого клиента;
2. какие общие паттерны поведения видны в группе;
3. чем кейсы отличаются друг от друга;
4. какую общую процессную проблему можно проверить;
5. какие выводы делать нельзя из-за ограничений данных.

Используй только переданные Markdown-отчеты.
</task>

<markdown_case_reports>
{{MARKDOWN_CASE_REPORTS}}
</markdown_case_reports>
""".strip()


# ============================================================
# 3. PROMPT ДЛЯ ФИНАЛЬНОГО ОБОБЩЕНИЯ БАТЧЕЙ
# ============================================================

FINAL_GROUP_SUMMARY_SYSTEM_PROMPT = """
<role>
Ты — ведущий аналитик антифрод-сработок и клиентского опыта.
Ты объединяешь несколько промежуточных групповых сводок в один итоговый отчет.
</role>

<task>
На вход переданы промежуточные групповые анализы, полученные по батчам кейсов.

Сформируй единый итоговый group_summary:
1. объедини повторяющиеся паттерны;
2. сохрани различия между сегментами;
3. укажи, какие проблемы подтверждаются несколькими батчами;
4. отдели факты от гипотез;
5. сформулируй общую процессную зону для проверки.
</task>

<strict_rules>
Используй только переданные batch summaries.

Не усиливай выводы.
Не превращай гипотезы в факты.
Если паттерн встречался только в одном батче, укажи это.
Если паттерн встречался в нескольких батчах, отметь его как более устойчивый.

Сохраняй ссылки на batch_id и case_id, если они есть в промежуточных сводках.
</strict_rules>

<output_format>
Верни Markdown без JSON-обертки и без markdown code fence.

Структура:

# Итоговый групповой анализ сработок

## 1. Краткий вывод

## 2. Основные повторяющиеся паттерны

Для каждого паттерна:
- описание;
- в каких батчах встречается;
- какие кейсы подтверждают;
- устойчивость;
- ограничения.

## 3. Проблемы клиентов по группе

Сводная таблица:
| Тип проблемы / наблюдения | Кейсы | Что подтверждено | Что не подтверждено |
|---|---|---|---|

## 4. Общие паттерны поведения до сработки

## 5. Общие паттерны поведения после сработки

## 6. Различия и подгруппы

## 7. Возможная процессная проблема для проверки

## 8. Ограничения интерпретации

## 9. Что проверить аналитически

## 10. Формулировка для итогового отчета руководителю
</output_format>

<style>
Пиши на русском языке.
Стиль — деловой, аналитический, осторожный.
</style>
""".strip()


FINAL_GROUP_SUMMARY_HUMAN_PROMPT = """
<task>
Объедини промежуточные групповые анализы в один итоговый group_summary.
</task>

<batch_summaries>
{{BATCH_SUMMARIES}}
</batch_summaries>
""".strip()


# ============================================================
# 4. ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
# ============================================================

def _extract_llm_text(response) -> str:
    """Достает текст из ответа LLM.

    Args:
        response: Ответ модели в виде строки, dict, LangChain AIMessage или объекта с `content`.

    Returns:
        Текст ответа модели.
    """

    if isinstance(response, str):
        return response
    if isinstance(response, dict):
        if "content" in response:
            return response["content"]
        return str(response)
    if hasattr(response, "content"):
        return response.content
    return str(response)


def _make_case_ids(df: pd.DataFrame, case_id_col: str | None = None) -> list[str]:
    """Создает case_id для каждого отчета.

    Args:
        df: DataFrame с отчетами.
        case_id_col: Название колонки с идентификатором кейса или None.

    Returns:
        Список строковых case_id.
    """

    if case_id_col is not None and case_id_col in df.columns:
        return df[case_id_col].astype(str).tolist()
    return [f"case_{i + 1:03d}" for i in range(len(df))]


def wrap_markdown_cases(case_markdowns: list[str], case_ids: list[str] | None = None) -> str:
    """Оборачивает Markdown-отчеты в XML-like case-блоки.

    Args:
        case_markdowns: Список Markdown-отчетов по кейсам.
        case_ids: Список идентификаторов кейсов или None.

    Returns:
        Строка с блоками `<case id="...">...</case>`.
    """

    if case_ids is None:
        case_ids = [f"case_{i + 1:03d}" for i in range(len(case_markdowns))]
    if len(case_ids) != len(case_markdowns):
        raise ValueError("len(case_ids) должен совпадать с len(case_markdowns)")

    blocks = []
    for case_id, markdown_text in zip(case_ids, case_markdowns):
        text = "" if pd.isna(markdown_text) else str(markdown_text).strip()
        blocks.append(f'<case id="{case_id}">\n{text}\n</case>')
    return "\n\n".join(blocks)


def build_group_summary_messages(markdown_case_reports: str) -> list[dict]:
    """Формирует messages для анализа одной группы Markdown-отчетов.

    Args:
        markdown_case_reports: Markdown-отчеты, обернутые в `<case>`-блоки.

    Returns:
        Список сообщений для LLM.
    """

    return [
        {"role": "system", "content": GROUP_SUMMARY_SYSTEM_PROMPT},
        {
            "role": "user",
            "content": GROUP_SUMMARY_HUMAN_PROMPT.replace("{{MARKDOWN_CASE_REPORTS}}", markdown_case_reports),
        },
    ]


def build_final_group_summary_messages(batch_summaries: str) -> list[dict]:
    """Формирует messages для финального объединения batch summaries.

    Args:
        batch_summaries: Текст промежуточных batch summaries.

    Returns:
        Список сообщений для LLM.
    """

    return [
        {"role": "system", "content": FINAL_GROUP_SUMMARY_SYSTEM_PROMPT},
        {
            "role": "user",
            "content": FINAL_GROUP_SUMMARY_HUMAN_PROMPT.replace("{{BATCH_SUMMARIES}}", batch_summaries),
        },
    ]


def split_df_into_batches(df: pd.DataFrame, batch_size: int) -> list[pd.DataFrame]:
    """Делит DataFrame на батчи.

    Args:
        df: DataFrame для разбиения.
        batch_size: Размер одного батча.

    Returns:
        Список DataFrame-батчей.
    """

    if batch_size <= 0:
        raise ValueError("batch_size должен быть больше 0")
    return [df.iloc[start:start + batch_size].copy() for start in range(0, len(df), batch_size)]


# ============================================================
# 5. ГЛАВНАЯ ФУНКЦИЯ: DF[case_report] -> group_summary
# ============================================================

def build_group_summary_from_case_reports(
    df: pd.DataFrame,
    llm,
    case_report_col: str = CASE_REPORT_COL,
    case_id_col: str | None = CASE_ID_COL,
    batch_size: int = BATCH_SIZE,
) -> tuple[str, pd.DataFrame]:
    """Строит group_summary по DataFrame с готовыми case_report.

    Args:
        df: DataFrame с готовыми отчетами по кейсам.
        llm: LLM-объект с методом `invoke(messages)`.
        case_report_col: Колонка с Markdown-отчетом по кейсу.
        case_id_col: Колонка с идентификатором кейса или None.
        batch_size: Количество case_report в одном батче.

    Returns:
        Кортеж `(group_summary, batch_summaries_df)`, где `group_summary` — итоговый Markdown,
        а `batch_summaries_df` — промежуточные сводки по батчам.
    """

    if case_report_col not in df.columns:
        raise ValueError(f"В df нет колонки {case_report_col!r}")

    work_df = df.copy()
    work_df = work_df[work_df[case_report_col].notna()].copy()
    work_df = work_df[work_df[case_report_col].astype(str).str.strip() != ""].copy()
    work_df = work_df.reset_index(drop=True)

    if len(work_df) == 0:
        raise ValueError("В df нет непустых case_report")

    work_df["_case_id_for_summary"] = _make_case_ids(work_df, case_id_col)
    batches = split_df_into_batches(work_df, batch_size=batch_size)
    batch_results = []

    for batch_num, batch_df in enumerate(tqdm(batches, desc="LLM batch group summaries"), start=1):
        batch_id = f"batch_{batch_num:03d}"
        markdown_case_reports = wrap_markdown_cases(
            case_markdowns=batch_df[case_report_col].tolist(),
            case_ids=batch_df["_case_id_for_summary"].tolist(),
        )
        response = llm.invoke(build_group_summary_messages(markdown_case_reports))
        batch_summary = _extract_llm_text(response)
        batch_results.append(
            {
                "batch_id": batch_id,
                "case_count": len(batch_df),
                "case_ids": batch_df["_case_id_for_summary"].tolist(),
                "batch_summary": batch_summary,
            }
        )

    batch_summaries_df = pd.DataFrame(batch_results)

    if len(batch_summaries_df) == 1:
        return batch_summaries_df.loc[0, "batch_summary"], batch_summaries_df

    batch_summaries_text_parts = []
    for _, row in batch_summaries_df.iterrows():
        batch_summaries_text_parts.append(
            f'<batch_summary id="{row["batch_id"]}" case_count="{row["case_count"]}">\n'
            f'Кейсы: {", ".join(row["case_ids"])}\n\n'
            f'{row["batch_summary"]}\n'
            f'</batch_summary>'
        )

    final_response = llm.invoke(build_final_group_summary_messages("\n\n".join(batch_summaries_text_parts)))
    return _extract_llm_text(final_response), batch_summaries_df


def build_group_summaries_for_groups(
    df: pd.DataFrame,
    llm,
    group_col: str = "group_name",
    case_report_col: str = CASE_REPORT_COL,
    case_id_col: str | None = CASE_ID_COL,
    batch_size: int = BATCH_SIZE,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Строит group_summary отдельно для каждой группы.

    Args:
        df: DataFrame с колонками группы и case_report.
        llm: LLM-объект с методом `invoke(messages)`.
        group_col: Название колонки группы.
        case_report_col: Название колонки с Markdown-отчетом по кейсу.
        case_id_col: Колонка с идентификатором кейса или None.
        batch_size: Количество case_report в одном батче.

    Returns:
        Кортеж `(group_summaries_df, all_batch_summaries_df)`.
    """

    if group_col not in df.columns:
        raise ValueError(f"В df нет колонки группы {group_col!r}")

    group_results = []
    batch_parts = []
    for group_name, group_df in tqdm(df.groupby(group_col, dropna=False), desc="LLM group summaries"):
        try:
            group_summary, batch_summaries_df = build_group_summary_from_case_reports(
                df=group_df,
                llm=llm,
                case_report_col=case_report_col,
                case_id_col=case_id_col,
                batch_size=batch_size,
            )
            group_results.append(
                {
                    group_col: group_name,
                    "group_summary": group_summary,
                    "group_summary_error": None,
                }
            )
            batch_summaries_df = batch_summaries_df.copy()
            batch_summaries_df[group_col] = group_name
            batch_parts.append(batch_summaries_df)
        except Exception as error:
            group_results.append(
                {
                    group_col: group_name,
                    "group_summary": None,
                    "group_summary_error": str(error),
                }
            )

    all_batch_summaries_df = pd.concat(batch_parts, ignore_index=True) if batch_parts else pd.DataFrame()
    return pd.DataFrame(group_results), all_batch_summaries_df

### 13.2 Run group summaries

In [ ]:
# ============================================================
# 6. ЗАПУСК
# ============================================================
#
# Вариант A. Проверить одну группу вручную:
#
# df = row_reports_df[row_reports_df["group_name"] == "нужная группа"].copy()
# llm = globals()[LLM_CLIENT_NAME]
# group_summary, batch_summaries_df = build_group_summary_from_case_reports(
#     df=df,
#     llm=llm,
#     case_report_col="case_report",
#     case_id_col=None,
#     batch_size=8,
# )
# print(group_summary)
#
# Вариант B. Запустить для всех group_name в row_reports_df:
#   RUN_LLM_GROUP_SUMMARIES = True
# ============================================================

if RUN_LLM_GROUP_SUMMARIES:
    llm = globals().get("llm", globals()[LLM_CLIENT_NAME])
    group_summaries_df, batch_summaries_df = build_group_summaries_for_groups(
        df=row_reports_df,
        llm=llm,
        group_col="group_name",
        case_report_col="case_report",
        case_id_col=None,
        batch_size=BATCH_SIZE,
    )
else:
    print("RUN_LLM_GROUP_SUMMARIES=False: LLM не вызван.")
    group_summaries_df = row_reports_df[["group_name"]].drop_duplicates().copy()
    group_summaries_df["group_summary"] = None
    group_summaries_df["group_summary_error"] = "LLM group summaries не запускались."
    batch_summaries_df = pd.DataFrame(columns=["batch_id", "case_count", "case_ids", "batch_summary", "group_name"])

show_df(group_summaries_df, "16. group_summaries_df", rows=10)
show_df(batch_summaries_df, "16.1 batch_summaries_df", rows=10)

## 14. SAVE FINAL Excel

In [ ]:
final_df = external_enriched_df.copy().reset_index().rename(columns={"index": "row_index"})
final_df = final_df.merge(
    row_reports_df[["row_index", "case_summary", "case_report", "llm_error"]],
    on="row_index",
    how="left",
)
final_df = final_df.merge(
    group_summaries_df[["group_name", "group_summary", "group_summary_error"]],
    on="group_name",
    how="left",
)

preferred_columns = [
    "group_name", "group_summary", "case_summary", "case_report", "llm_error",
    "epk_id", "event_dt", "event_id", "answer_date", "answer_1", "comment_8",
    "channel", "event_channel", "sub_channel", "event_type", "transaction_amount",
    "mcc_code", "mcc_name", "description", "resolution_last", "rule_name",
]
ordered_columns = [column for column in preferred_columns if column in final_df.columns]
final_df = final_df[ordered_columns + [column for column in final_df.columns if column not in ordered_columns]]

write_table_file(final_df, FINAL_REPORT_PATH)
show_df(final_df, "17. final_df", rows=3)
print("Final Excel:", FINAL_REPORT_PATH.resolve())
